# Getting Started with DNN

Before we move to a more sophisticated ex. we will get familiar with
Tensorflow and Keras. We will take a look at a first example of
a neural network, which makes use of the Python library Keras to learn
to classify hand-written digits. The problem we are trying to solve here
is to classify grayscale images of handwritten digits (28 pixels by 28
pixels), into their 10 categories (0 to 9). The dataset we will use is
the MNIST dataset. It's a set of 60,000 training images, plus 10,000
test images, assembled by the National Institute of Standards and
Technology (the NIST in MNIST) in the 1980s. You can think of "solving"
MNIST as the "Hello World" of deep learning -- it's what you do to
verify that your algorithms are working as expected.

Firstly, we will import Keras. Historically, there were many ways to do it:

If you are using the newest version of Keras 3 you can use it with different backends instead of just Tensorflow. Keras now supports both PyTorch and Jax as well. You should specify the backend you want, before importing Keras. Or just use a default tensorflow.

In [2]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import keras
print(keras.__version__)

3.16.0.dev2026100704


The MNIST dataset comes pre-loaded in Keras, in the form of a set of
four Numpy arrays:

In [3]:
from keras.datasets import mnist

(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

**DNN workflow**
- Build the neural network architecture.
- Train our neural network with the training data, `train_images` and
  `train_labels`. The network will then learn to associate images and
  labels.
- The network will produce predictions for `test_images`, and we will
  verify if these predictions match the labels from `test_labels`.

**DNN architecture**

- Our network will consist of a sequence of two Dense layers, which are
densely-connected (also called "fully-connected") neural layers.

- The second (and last) layer is a 10-way "softmax" layer, which means
  it will return an array of 10 probability scores (summing to 1). Each
  score will be the probability that the current digit image belongs to
  one of our 10 digit classes.

**DNN training**

To make our network ready for training, we need to pick three more
things, as part of the "compilation" step:

- A **loss function**: This is how the network will be able to measure
how good a job it is doing on its training data, and thus how it will be
able to steer itself in the right direction.
- An **optimizer**: this is the mechanism through which the network will
  update itself based on the data it sees and its loss function.
- **Metrics**: to monitor during training and testing. Here we will only
  care about accuracy (the fraction of the images that were correctly
  classified).

Today, during our 3 examples we will use only the `Sequential class`.
During our next meetings, I will introduce the functional API where we
will be able to manipulate the data tensors that the model processes and
apply layers to this tensors as if they were functions.




# Network architecture

In [4]:
from keras import models
from keras import layers

network = models.Sequential()

network.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
network.add(layers.MaxPooling2D((2, 2)))
network.add(layers.Conv2D(64, (3, 3), activation='relu'))
network.add(layers.MaxPooling2D((2, 2)))
network.add(layers.Flatten())

network.add(layers.Dense(128, activation='relu'))
network.add(layers.Dense(10, activation='softmax'))

#Instead of input_shape parameter, you can use an Input layer

C:\Users\piotr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Network Training

Once the model architecture is defined, the learning process can be
configured in the compilation step. We specify the optimizer, loss
function, and metrics.

In [5]:
network.compile(optimizer='rmsprop',
                loss='categorical_crossentropy',
                metrics=['accuracy'])

**Data preparation**\
Before training, we will preprocess our data by reshaping it into the
shape that the network expects, and scaling it so that all values are in
the \[0, 1\] interval. Our training images are being stored in an array
of shape (60000, 28, 28) of type uint8 with values in the \[0, 255\]
interval. Please transform it into a float32 array of shape (60000, 28
\* 28) with values between 0 and 1.

Necessary functions: `reshape` and `astype`.

Perform it for both train and test examples.

In [6]:
train_images_reshaped = train_images.astype('float32') / 255
test_images_reshaped = test_images.astype('float32') / 255

Because of we use `categorical_crossentropy` loss function we need to
convert data format:

In [7]:
from keras.utils import to_categorical

train_labels_cat = to_categorical(train_labels)
test_labels_cat = to_categorical(test_labels)

**Fit the model**\
To train the network we call the `fit` method of the network with
parameters `epochs` and `batch_size`. Set `epochs` to 5, and
`batch_size` to 128.

In [8]:
network.fit(train_images_reshaped, train_labels_cat, batch_size=128, epochs=5)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.9415 - loss: 0.1883
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 8s 16ms/step - accuracy: 0.9837 - loss: 0.0515
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 16ms/step - accuracy: 0.9900 - loss: 0.0335
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9930 - loss: 0.0236
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9954 - loss: 0.0173


## Network evaluation

Two quantities are being displayed during training: the "loss" of the
network over the training data, and the accuracy of the network over the
training data. We quickly reach an accuracy of 0.989 (i.e. 98.9%) on the
training data. Now let's check that our model performs well on the test
set too:

In [9]:
test_loss, test_acc = network.evaluate(test_images_reshaped, test_labels_cat)

print('test_acc:', test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9880 - loss: 0.0451
test_acc: 0.9879999756813049


Our test set accuracy turns out to be 97.8% -- that's quite a bit lower
than the training set accuracy.\
This gap between training accuracy and test accuracy is an example of
"overfitting", the fact that machine learning models tend to perform
worse on new data than on their training data.

In [10]:
network.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 450,070 (1.72 MB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 225,036 (879.05 KB)